# Football Player Clustering — Corrected Workflow

This notebook follows the structure and column names of the supplied `Clustering_Player_English_Explained.ipynb`. It creates separate player profiles for defenders (DF), midfielders (MF), and forwards (FW).

**Important:** Place `Final_Dataset.csv` in the same folder as this notebook before running it. The original notebook was supplied without that CSV, so the data-dependent results cannot be verified here. The workflow below checks required columns and reports clear errors rather than silently changing the dataset.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RANDOM_STATE = 42


In [ ]:
# Load the same source file referenced by the original notebook.
# index_col=0 preserves the original convention used in that notebook.
df = pd.read_csv("Final_Dataset.csv", index_col=0)

# Remove accidental whitespace from column names, without renaming the dataset fields.
df.columns = df.columns.astype(str).str.strip()

required_identity = ["Player", "Team", "Position"]
missing_identity = [c for c in required_identity if c not in df.columns]
if missing_identity:
    raise ValueError(f"Final_Dataset.csv is missing required columns: {missing_identity}")

# Create a stable row key. This prevents incorrect merges when a player/team pair
# occurs more than once in the source data.
df = df.reset_index(drop=True).copy()
df["_row_id"] = np.arange(len(df))

# The source file may store Age as text such as '23-...' or as a number.
if "Age" in df.columns:
    age_text = df["Age"].astype("string").str.strip()
    df["Age"] = pd.to_numeric(age_text.str.extract(r"(\d{1,2})", expand=False), errors="coerce")

# Match the original broad position grouping: DF, MF, and FW.
df["Position"] = df["Position"].astype("string").str.strip().str[:2].str.upper()
df[["Player", "Team", "Position"]].head()


## Features used for clustering

The following lists retain the statistical features used in the supplied notebook. Features are selected separately for each position group. If a listed column is absent from the CSV, the notebook stops and tells you which source columns need attention.

In [ ]:
# Original feature lists, retained from the supplied notebook.
feature_sets = {
    "DF": [
        "Goals", "Ast", "xA", "Through Balls", "Switch Passes", "Crosses",
        "Shot Creating Actions", "Def Action to Shot", "Tackles", "Shots Blocked",
        "Interceptions", "Clearances", "Dribbles Successful", "Dribble Distance",
        "Ball Recoveries", "Air Duels Won%"
    ],
    "MF": [
        "Goals", "Ast", "xA", "Through Balls", "Switch Passes", "Crosses",
        "Shot Creating Actions", "Def Action to Shot", "Tackles", "Shots Blocked",
        "Interceptions", "Clearances", "Dribbles Successful", "Dribble Distance",
        "Ball Recoveries", "Air Duels Won%"
    ],
    "FW": [
        "Goals", "xG", "xA", "Pass Cmp%", "SoT%", "Ast", "Through Balls",
        "Switch Passes", "Crosses", "Shot Creating Actions", "Def Action to Shot",
        "Tackles", "G-xG", "Interceptions", "Dribbles Successful",
        "Dribble Distance", "Ball Recoveries", "Air Duels Won%"
    ]
}

for pos, features in feature_sets.items():
    absent = [c for c in features if c not in df.columns]
    if absent:
        raise ValueError(f"Missing {pos} clustering feature columns in Final_Dataset.csv: {absent}")

position_counts = df["Position"].value_counts(dropna=False)
print("Rows by position code:")
display(position_counts)


## Preprocessing and clustering function

For each position group, missing feature values are imputed with that feature's **median**, then features are standardized before PCA. PCA retains enough components to explain at least **95% of the variance** for clustering. A separate two-component PCA is used only for the scatter plot, so the clustering does not discard information merely to make a 2D chart.

The number of clusters is evaluated over the feasible range 2–10 using both inertia (Elbow method) and silhouette score. The highest silhouette score selects the working K; cluster IDs are statistical labels, not predefined football roles.

In [ ]:
def cluster_position(data, position_code, feature_cols):
    """Impute, standardize, PCA-reduce, evaluate K, and cluster one position group."""
    subset = data.loc[data["Position"] == position_code].copy().reset_index(drop=True)
    if len(subset) < 3:
        raise ValueError(f"Not enough {position_code} players ({len(subset)}) for clustering; at least 3 are required.")

    # Convert selected fields to numeric; non-numeric entries become missing.
    X_raw = subset[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Entirely empty columns cannot be median-imputed; remove them explicitly.
    usable_features = X_raw.columns[X_raw.notna().any()].tolist()
    removed_features = [c for c in feature_cols if c not in usable_features]
    if len(usable_features) < 2:
        raise ValueError(f"At least two usable numeric features are required for {position_code}; found {len(usable_features)}.")
    if removed_features:
        print(f"{position_code}: excluded all-missing feature columns: {removed_features}")
    X_raw = X_raw[usable_features]

    imputer = SimpleImputer(strategy="median")
    X_imputed = imputer.fit_transform(X_raw)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)

    # PCA for clustering: retain 95% of standardized feature variance.
    pca_full = PCA(n_components=0.95, svd_solver="full")
    X_pca = pca_full.fit_transform(X_scaled)

    # Feasible K values require at least one more sample than clusters.
    max_k = min(10, len(subset) - 1)
    k_values = list(range(2, max_k + 1))
    if not k_values:
        raise ValueError(f"Not enough {position_code} players to evaluate K-Means.")

    inertia_values, silhouette_values = [], []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = model.fit_predict(X_pca)
        inertia_values.append(model.inertia_)
        distinct_labels = np.unique(labels).size
        if 2 <= distinct_labels <= len(subset) - 1:
            silhouette_values.append(silhouette_score(X_pca, labels))
        else:
            # Degenerate solutions can occur when rows are identical.
            silhouette_values.append(np.nan)

    valid_scores = np.asarray(silhouette_values, dtype=float)
    if np.isnan(valid_scores).all():
        raise ValueError(f"Silhouette evaluation was undefined for every K in {position_code}; inspect duplicate or constant player records.")

    # Silhouette score is a reproducible selection criterion; inspect both plots too.
    best_k = k_values[int(np.nanargmax(valid_scores))]
    final_model = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
    cluster_labels = final_model.fit_predict(X_pca)

    # A distinct 2D PCA is used only to visualize the full standardized feature space.
    pca_2d = PCA(n_components=2)
    coords = pca_2d.fit_transform(X_scaled)
    result = subset[["_row_id", "Player", "Team", "Position"]].copy()
    result["PC1"] = coords[:, 0]
    result["PC2"] = coords[:, 1]
    result["cluster"] = cluster_labels

    # Evaluation plots
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(k_values, inertia_values, marker="o")
    axes[0].set(title=f"{position_code}: Elbow method", xlabel="Number of clusters (K)", ylabel="Inertia")
    axes[0].set_xticks(k_values)
    axes[1].plot(k_values, silhouette_values, marker="o")
    axes[1].set(title=f"{position_code}: Silhouette scores", xlabel="Number of clusters (K)", ylabel="Silhouette score")
    axes[1].set_xticks(k_values)
    plt.tight_layout()
    plt.show()

    print(f"{position_code}: {len(subset)} players | {len(usable_features)} usable features")
    print(f"PCA components for clustering: {X_pca.shape[1]} | cumulative explained variance: {pca_full.explained_variance_ratio_.sum():.3f}")
    print(f"Selected K: {best_k} | silhouette score: {np.nanmax(silhouette_values):.3f}")

    fig = px.scatter(
        result, x="PC1", y="PC2", color=result["cluster"].astype(str),
        hover_data=["Player", "Team"], title=f"{position_code}: Player clusters (2D PCA visualization)"
    )
    fig.update_layout(coloraxis_colorbar_title="Cluster")
    fig.show()

    return result, {
        "position": position_code,
        "players": len(subset),
        "features_used": usable_features,
        "pca_components_95pct": X_pca.shape[1],
        "explained_variance": float(pca_full.explained_variance_ratio_.sum()),
        "k_values": k_values,
        "silhouette_scores": silhouette_values,
        "selected_k": best_k,
        "selected_silhouette": float(np.nanmax(silhouette_values)),
    }


In [ ]:
# Run the same workflow separately for defenders, midfielders, and forwards.
cluster_results = {}
evaluation_results = []

for position_code in ["DF", "MF", "FW"]:
    if (df["Position"] == position_code).sum() == 0:
        print(f"Skipping {position_code}: no rows found in the source dataset.")
        continue
    labels_df, evaluation = cluster_position(df, position_code, feature_sets[position_code])
    cluster_results[position_code] = labels_df
    evaluation_results.append(evaluation)

evaluation_summary = pd.DataFrame([
    {k: v for k, v in item.items() if k not in ["features_used", "k_values", "silhouette_scores"]}
    for item in evaluation_results
])
display(evaluation_summary)


## Understand the cluster profiles

Cluster numbers are assigned by K-Means and can change if the data changes. Do **not** attach fixed labels such as “Target Man” or “Deep-Lying Playmaker” to a cluster number without inspecting its actual statistics. The tables below show mean values in the original feature units, making it possible to interpret each cluster from the supplied data.

In [ ]:
# Join the assigned labels to the untouched source rows using the stable row key.
if not cluster_results:
    raise RuntimeError("No position groups were clustered; check the Position values in the source CSV.")

labels_all = pd.concat(cluster_results.values(), ignore_index=True)
final_df = df.merge(
    labels_all[["_row_id", "cluster", "PC1", "PC2"]],
    on="_row_id", how="left", validate="one_to_one"
)

# Cluster profile summaries use original, unscaled feature values.
profile_tables = {}
for pos, features in feature_sets.items():
    pos_rows = final_df.loc[final_df["Position"] == pos]
    if pos_rows.empty or "cluster" not in pos_rows:
        continue
    available = [c for c in features if c in pos_rows.columns]
    profile = pos_rows.groupby("cluster")[available].mean(numeric_only=True).round(3)
    profile_tables[pos] = profile
    print(f"\n{pos} cluster profiles (mean original feature values):")
    display(profile)


## Final checks and CSV export

The export keeps all original player columns and adds `cluster`, `PC1`, and `PC2`. The `_row_id` helper is removed from the deliverable. Unclustered position groups (for example, goalkeepers) remain in the output with missing cluster values rather than being silently discarded.

In [ ]:
# Basic integrity checks before writing the output.
assert len(final_df) == len(df), "Final row count changed unexpectedly."
assert final_df["_row_id"].is_unique, "Row key is not unique; inspect the source data."

# Remove the internal matching key from the exported dataset.
final_export = final_df.drop(columns=["_row_id"])
output_path = "Clusters.csv"
final_export.to_csv(output_path, index=False)

print(f"Saved {len(final_export)} rows to {output_path}")
print("Rows with assigned clusters:", int(final_export["cluster"].notna().sum()))
print("Rows without clusters (for example, other positions):", int(final_export["cluster"].isna().sum()))
print("Cluster counts by position:")
display(final_export.groupby("Position", dropna=False)["cluster"].value_counts(dropna=False).rename("players").reset_index())
